In [1]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub
from agentic.mcp_servers.udahub.knowledge_index import KnowledgeIndex, create_udahub_engine

# Udahub Application

## Core Database

**Init DB**

In [2]:
udahub_db = "data/core/udahub.db"

In [3]:
reset_db(udahub_db)

✅ Removed existing data/core/udahub.db
2026-11-22 22:14:13,863 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-11-22 22:14:13,865 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/core/udahub.db with fresh schema


In [4]:
engine = create_udahub_engine(udahub_db)  # also loads sqlite-vec
udahub.Base.metadata.create_all(bind=engine)

**Account**

In [5]:
account_id = "cultpass"
account_name = "CultPass Card"

In [6]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

**Knowledge Base**

In [7]:
# TODO: Create additional 10 articles  (Rubric 1: >= 14 total, varied categories)
# NOTE: data/external/cultpass_articles.jsonl currently has 15 records, so the count
#       check below should already pass. Still verify the *variety* requirement:
#
# pseudocode:
#   load all articles
#   for each article: split its `tags` string into a set of tags
#   map tags -> category in {technical, billing, account management, reservations, ...}
#   assert every required category has >= 1 article
#   if a category is missing: append new {"title","content","tags"} lines to the .jsonl
#   print the category counts so a reviewer can see the variety

In [8]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

In [9]:
cultpass_articles

[{'title': 'How to Reserve a Spot for an Event',
  'content': 'If a user asks how to reserve an event:\n\n- Guide them to the CultPass app\n- Instruct them to browse the experience catalog and tap \'Reserve\'\n- If it\'s a premium or limited event, check if reservation confirmation is required via email\n- Remind them to arrive at least 15 minutes early with their QR code visible\n\n**Suggested phrasing:**\n"You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping \'Reserve\'. Be sure to arrive 15 minutes early with your QR code ready."',
  'tags': 'reservation, events, booking, attendance'},
 {'title': "What's Included in a CultPass Subscription",
  'content': 'Each user is entitled to 4 cultural experiences per month, which may include:\n- Art exhibitions\n- Museum entries\n- Music concerts\n- Film screenings and more\n\nSome premium experiences may require an additional fee (visible in the app).\n\n**Suggested phrasing:**\n"Your CultPass s

In [10]:
if len(cultpass_articles) < 14:
    raise AssertionError("You should load the articles with at least 14 records")

In [11]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb) 
    

**Knowledge index (sqlite-vec)**

In [12]:
# Embeds every article (OpenAI embeddings via .env) and stores the vectors in vec_knowledge.
# Re-run this cell whenever the knowledge table changes. reset_db() above wipes the index too.
knowledge_index = KnowledgeIndex(engine)
print(f"Indexed {knowledge_index.rebuild()} articles")

Indexed 15 articles


In [13]:
for question in [
    "I can't log in to my account",
    "my app keeps crashing when I open it",
    "how do I get my money back",
    "what's the weather like on mars",
]:
    result = knowledge_index.search(question, k=3)
    print(question, "->", result["top_score"])
    for r in result["results"]:
        print(f"   {r['score']:.3f}  {r['title']}")

I can't log in to my account -> 0.4684
   0.468  How to Handle Login Issues?
   0.409  Account Recovery Process
   0.340  How to Update Profile Information
my app keeps crashing when I open it -> 0.5511
   0.551  CultPass App Not Opening
   0.454  Troubleshooting App Performance
   0.256  How to Provide Feedback or Report an Issue
how do I get my money back -> 0.3435
   0.344  Refund Policy Overview
   0.304  Troubleshooting Payment Issues
   0.264  Account Recovery Process
what's the weather like on mars -> 0.1037
   0.104  Managing Notification Settings
   0.086  How to Provide Feedback or Report an Issue
   0.077  Refund Policy Overview


**Ticket**

In [14]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [15]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [16]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )
    
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])


**Long-term memory**

In [17]:
# Long-term memory is the `customer_memory` table, created above with the other tables.
# The support orchestrator writes to it when a ticket is resolved (what was fixed, stated
# preferences such as language) and reads it back when the customer returns.
from sqlalchemy import inspect

print([column["name"] for column in inspect(engine).get_columns("customer_memory")])

['memory_id', 'account_id', 'user_id', 'memory_type', 'key', 'value', 'source_ticket_id', 'created_at', 'updated_at']


# Tests

In [18]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(account)

<Account(account_id='cultpass', account_name='CultPass Card')>


In [19]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    for article in account.knowledge_articles:
        print(article)

<Knowledge(article_id='90f492f1-46b0-4213-a071-24b00dbb6657', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='a90787ec-ed08-4c0a-8027-272d012c528b', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='db8cf695-59d0-47cc-b21a-45a66601b238', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='b47afbab-2710-4af0-83c3-da557d0c12d2', title='How to Handle Login Issues?')>
<Knowledge(article_id='0437c181-0e47-4263-82cb-187b3e7fa501', title='CultPass App Not Opening')>
<Knowledge(article_id='0f6ccbe8-2867-4813-a8d6-b297f47110c2', title='Troubleshooting App Performance')>
<Knowledge(article_id='c936b683-2b61-44bc-a251-620f247cd3fa', title='Managing Notification Settings')>
<Knowledge(article_id='4afd25d3-a37a-4785-8900-d05e3bcf5961', title='Refund Policy Overview')>
<Knowledge(article_id='a73664b2-651b-4ff9-aabd-33cd139e138c', title='Contacting CultPass Support')>
<Knowledge(article_id='0b2d0087-f0c6-4b50-9c90-74bb842e128b', title='

In [20]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='c0bd1852-d12b-47b6-8465-9c450b8e7919', user_name='Alice Kingsley', external_user_id='a4ab87')>


In [21]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    
    ticket:udahub.Ticket = user.tickets[0]
    for message in ticket.messages:
        print(message)

<TicketMessage(message_id='dc8e1ecc-a0aa-4b91-81c6-1ae245932367', role='user', content='I can't log in to my Cultpass ...')>
